## 1. Установка google disk

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

domain_path = '/content/drive/MyDrive/Colab Notebooks/graphRAG'
os.chdir(domain_path)
print(os.listdir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['RAG_example02.ipynb', 'prompts', 'domains', 'src']


In [15]:
import os

domain_name = "HPotter"
domain_path = f"./domains/{domain_name}"

# Проверяем, существует ли папка, если нет — создаем
os.makedirs(domain_path, exist_ok=True)

fileslist = os.listdir(domain_path)
fileslist

['RAG_Llama_Qwen_HP.ipynb', 'questions.xlsx', 'docs', 'markedocs']

## 2. Готовим модель и проверяем ее работу

In [7]:
import requests
import json

from typing import Optional, List, Dict
from openai import OpenAI
import json


class YandexGPTAgentClass:

    def __init__(self, config_dict: dict):
        """
        Initializes the LLMAgent with a configuration dictionary.
        Expected config_dict keys: 'base_URL', 'api_key', 'base_model', 'model_config', 'cloud_folder_id'.
        """
        self.base_url = config_dict.get('base_URL')
        self.api_key = config_dict.get('api_key')
        self.model_name = config_dict.get('model_name')
        self.model_config = config_dict.get('model_config', {})
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key)
        # этот параметр характерен именно для вызова YaGPT моделей
        self.cloud_folder_id = config_dict.get('cloud_folder_id') # Добавлен cloud_folder_id
        if not self.cloud_folder_id:
            raise ValueError("YandexGPTAgent is not fully configured. Missing cloud_folder_id.")


    def invoke(self, messages: List[Dict[str, str]], **kwargs) -> str:
        """
        Runs a request to the Yandex GPT API using the OpenAI client,
        adding Yandex-specific headers (X-Folder-ID).

        Args:
            messages (List[Dict[str, str]]): A list of message dictionaries, each with 'role' and 'text' keys.
            **kwargs: Additional keyword arguments specific to the LLM API call (e.g., temperature).

        Returns:
            str: The response text from the LLM.

        Raises:
            ValueError: If the agent is not fully configured.
            Exception: For general errors during API invocation.
        """
        if not self.base_url or not self.api_key or not self.cloud_folder_id:
            raise ValueError("YandexGPTAgent is not fully configured. Missing base_url, api_key, or cloud_folder_id.")

        completion_options = {
            "temperature": self.model_config.get("temperature", 0.6),
            "max_tokens": self.model_config.get("max_tokens", 25000)
        }
        for key in ['temperature', 'max_tokens', 'seed']: # Allow kwargs to override
            if key in kwargs:
                completion_options[key] = kwargs[key]

        # extra_headers = {"X-Folder-ID": self.cloud_folder_id}

        try:
            chat_completion = self.client.chat.completions.create(
                model=f"gpt://{self.cloud_folder_id}/{self.model_name}",
                messages=messages,
                # extra_headers=extra_headers,
                response_format=kwargs.get("response_format", {"type": "json_object"}), # Allow overriding response_format
                **completion_options
            )
            llm_output_str = chat_completion.choices[0].message.content
            return llm_output_str

        except Exception as e:
            raise Exception(f"Произошла ошибка при вызове YandexGPTAgent: {e}")

    def set_model(self, model_name: str, model_config: Optional[dict] = None):
        """
        Changes the base model name and/or updates the model parameters.

        Args:
            model_name (str): The new name of the model to use.
            model_config (Optional[dict]): A dictionary of new model parameters to update.
                                        If a key exists, its value will be updated; otherwise, it will be added.
        """
        if model_name:
            self.model_name = model_name
        if model_config is not None:
            # Ensure model_config exists in self.config before updating
            if self.model_config is None:
                self.model_config = {}
            self.model_config.update(model_config)

In [8]:
YANDEX_API_KEY = os.getenv('YANDEX_API_KEY', '')  # yandex API
os.environ['YANDEX_API_KEY'] = YANDEX_API_KEY

# задаем параметры облачных моделей yandex AI-Studio
YANDEX_CLOUD_FOLDER = "b1grdf3477jnqvmbh0n7"
os.environ['YANDEX_CLOUD_FOLDER'] = YANDEX_CLOUD_FOLDER

In [10]:
YANDEX_BASE_URL = "https://ai.api.cloud.yandex.net/v1"  # OpenAI-совместимый базовый URL для YandexGPT API

YA_LLM_config = {
    "config_name": "DOCKER_LLM_config", # название конфигурации
    'base_URL': YANDEX_BASE_URL,
    "api_key": os.getenv('YANDEX_API_KEY'),
    "model_name": "yandexgpt-5.1/latest", # Название используемой модели по умолчанию
    "cloud_folder_id": os.getenv('YANDEX_CLOUD_FOLDER'), # Идентификатор каталога для X-Folder-ID
    "model_config": {"max_tokens": 30000}, # параметры модели по умолчанию
}

yandex_agent = YandexGPTAgentClass(YA_LLM_config)
print("YandexGPTAgent инстанциирован успешно.")

YandexGPTAgent инстанциирован успешно.


In [11]:
# выберем модель LLM
model_name = "deepseek-v4-flash/latest"
model_name = "qwen3-235b-a22b-fp8/latest"
model_name = "gpt-oss-120b/latest"

model_config = {"max_tokens": 30000}
yandex_agent.set_model(model_name, model_config)

In [12]:
# тестим агента

dialog_prompt = [
    {"role": "system", "content": "You are a helpful assistant for knowledge base creation."},
    {"role": "user", "content": "Кто написал книгу про гарри Поттера ?"}
]

try:
    response_text = yandex_agent.invoke(dialog_prompt)
    print("YandexGPTAgent Ответ:")
    print(response_text)
except Exception as e:
    print(f"Произошла ошибка при вызове YandexGPTAgent: {e}")

YandexGPTAgent Ответ:
{
  "answer": "Книгу о Гарри Поттере написала британская писательница Джоан Кэтлин Роулинг, более известная под псевдонимом J.K. Rowling."
}


## 3. Прогоняем ответы на вопросы до RAG

In [13]:
import re
import pandas as pd


# Функция для удаления лишних символов
def clean_response(text):
    # Убираем все маркеры типа [/] и любые специальные символы
    clean_text = re.sub(r"\[\/?\w+\]", "", text)  # Убираем [SYS], [USER], и другие токены
    clean_text = re.sub(r"<\|[^>]*\|>", "", clean_text)  # Убираем токены вроде <|start_header_id|>
    return clean_text.strip()  # Убираем пробелы в начале и конце

In [ ]:
import os

question_file_path = os.path.join(domain_path, 'questions.xlsx')


# Чтение Excel-файла
df = pd.read_excel(question_file_path)

curr_model = model_name
# Добавление новых колонок для ответов модели
df[curr_model] = ''
promt = "Ты эксперт вселенной Гарри Поттера из романа Джоан Роулинг 'Гарри Поттер и философский камень'. Отвечай по-русски и строго по сюжету книги. Не придумывай. "
# Прогон вопросов через модель и запись ответов в df
for index, row in df.iterrows():
    question = promt + row['Вопрос']
    response = yandex_agent.invoke([{"role": "user", "content": question}])
    print(response)
    clean_answer = clean_response(response)
    df.at[index, curr_model] = clean_answer

# Сохранение DataFrame обратно в Excel
df.to_excel(os.path.join(domain_path, curr_model+'.xlsx'), index=False)

{ "answer": "Мантия Гриффиндорского дома в книге описана как красная (бордово‑красного оттенка) с золотыми элементами, соответствующими цветам дома – красному и золотому." }
{ "answer": "Палочка Гарри Поттера сделана из ольхи (holly) и имеет сердцевину из пера феникса — того же самого феникса Фоукса, принадлежащего Альбусу Дамблдору." }
{ "answer": "Самым популярным видом спорта среди волшебников, описанным в романе «Гарри Поттер и философский камень», является квиддич." }
{
  "answer": "Директора Хогвартса зовут Альбус Дамблдор."
}
{ "answer": "Профессор Снейп (Снегг) преподаёт предмет «Зельеварение»." }
{ "answer": "У Гарри в первой книге «Гарри Поттер и философский камень» используется метла модели **Nimbus 2000**. Она была подарена ему профессором Макгонагалл после того, как он был назначен поисковиком команды Гриффиндора." }
{ "answer": "Incendio", "explanation": "В книге «Гарри Поттер и философский камень» упоминается заклинание Incendio, которое используется для создания огня." 

## 4. Строим RAG

In [ ]:
def create_text_qa_template():
    """
    Создает шаблон для запросов Q&A.
    """
    chat_text_qa_msgs = [
        ChatMessage(
            role=MessageRole.SYSTEM,
            content=(
                "Ты эксперт вселенной Гарри Поттера из романа Джоан Роулинг 'Гарри Поттер и философский камень'. Отвечай по-русски и строго по контексту. Не придумывай."
            ),
        ),
        ChatMessage(
            role=MessageRole.USER,
            content=(
                "Контекстная информация приведена ниже.\n"
                "---------------------\n"
                "{context_str}\n"
                "---------------------\n"
                "Теперь ответь на вопрос с учетом контекста: {query_str}\n"
            ),
        ),
    ]
    text_qa_template = ChatPromptTemplate(chat_text_qa_msgs)
    return text_qa_template

In [ ]:
global_index = None
query_engine = None
def initialize_model():
    global global_index, query_engine

    embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-large")

    context_window = 5500
    num_output = 256

    # Загрузка локальной модели GGUF с помощью llama-cpp-python
    llm = LlamaCPP(
        #model_url="https://huggingface.co/IlyaGusev/saiga_llama3_8b_gguf/resolve/main/model-q4_K.gguf",
        #model_url="https://huggingface.co/IlyaGusev/saiga_llama3_8b_gguf/resolve/main/model-q8_0.gguf",
        model_url="https://huggingface.co/unsloth/Llama-3.2-3B-Instruct-GGUF/resolve/main/Llama-3.2-3B-Instruct-Q4_K_M.gguf",
        #model_url="https://huggingface.co/Qwen/Qwen2-7B-Instruct-GGUF/resolve/main/qwen2-7b-instruct-q4_0.gguf",
        temperature=0,
        max_new_tokens=100,
        context_window=context_window,
        generate_kwargs={},
        model_kwargs={"n_gpu_layers": -1},
        messages_to_prompt=messages_to_prompt,
        completion_to_prompt=completion_to_prompt,
        verbose=True,
    )

    chunk_size = 900 #600
    chunk_overlap = 256
    node_parser = SimpleNodeParser.from_defaults(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


    Settings.llm =llm
    Settings.embed_model=embed_model
    Settings.context_window=context_window
    Settings.num_output=num_output
    Settings.node_parser=node_parser


    documents = SimpleDirectoryReader('./RAG_data').load_data()

    global_index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

    text_qa_template = create_text_qa_template()

    query_engine = global_index.as_query_engine(
        similarity_top_k=5,
        text_qa_template=text_qa_template,
        return_source_documents=True  # Добавление этой опции для возврата контекста
    )

# Инициализация модели (это нужно выполнить один раз)
initialize_model()


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from /tmp/llama_index/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                       general.organization str              = Meta Llama
llama_model_loader: - kv   4:                           general.finetune str              = Instruct
llama_model_loader: - kv   5:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   6:                         general.size_label str              = 3B
llama_model_loader: - kv

In [ ]:
def clean_response(text):
    # Убираем все маркеры типа <<SYS>> и любые специальные символы
    clean_text = text.replace('<<SYS>>', '').replace('<</SYS>>', '').strip()
    clean_text = re.sub(r"\[\/?\w+\]", "", clean_text)  # Убираем [SYS], [USER], и другие токены
    clean_text = re.sub(r"<\|[^>]*\|>", "", clean_text)  # Убираем токены вроде <|start_header_id|>
    clean_text = re.sub(r"<\/?s>", "", clean_text)  # Убираем <s> и </s>
    clean_text = re.sub(r"assistant:\s*", "", clean_text)  # Убираем assistant: и возможные пробелы после

    # Обрезаем текст после первого предложения
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', clean_text)  # Находим первое полное предложение
    return sentences[0] if sentences else clean_text  # Возвращаем первое предложение

def make_rag(question):
    global query_engine
    response = query_engine.query(question)
    answer = response.response
    cleaned_answer = clean_response(answer)
    return cleaned_answer

In [ ]:
make_rag("Какой вид спорта является самым популярным среди волшебников?")

Llama.generate: 3062 prefix-match hit, remaining 1 prompt tokens to eval

llama_print_timings:        load time =     307.60 ms
llama_print_timings:      sample time =      16.90 ms /   100 runs   (    0.17 ms per token,  5916.46 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =    1967.28 ms /   100 runs   (   19.67 ms per token,    50.83 tokens per second)
llama_print_timings:       total time =    2082.35 ms /   100 tokens


'Волшебники играют в квиддич.'

In [ ]:
def make_rag_cntx(question):
    global query_engine
    response = query_engine.query(question)
    answer = response.response
    cleaned_answer = clean_response(answer)
    # Получение использованных источников (чанков)
    source_documents = response.source_nodes  # Эти узлы содержат чанки контекста
    chunks = [doc.node.text for doc in source_documents]  # Извлечение текста чанков

    # Выводим чанки, которые использовались для ответа
    for i, chunk in enumerate(chunks, 1):
        print(f"Чанк {i}: {chunk}")

    print(answer)
    return cleaned_answer

In [ ]:
response_with_chunks = make_rag_cntx("Где состоялась первая встреча Гарри и Квиррелла?")
print(response_with_chunks)

Llama.generate: 98 prefix-match hit, remaining 3050 prompt tokens to eval

llama_print_timings:        load time =     307.60 ms
llama_print_timings:      sample time =      16.66 ms /   100 runs   (    0.17 ms per token,  6003.12 tokens per second)
llama_print_timings: prompt eval time =    1986.95 ms /  3050 tokens (    0.65 ms per token,  1535.02 tokens per second)
llama_print_timings:        eval time =    1948.31 ms /    99 runs   (   19.68 ms per token,    50.81 tokens per second)
llama_print_timings:       total time =    4051.56 ms /  3149 tokens


Чанк 1: — воскликнул Гарри, не веря своим ушам.
   — Он всегда со мной, где бы я ни был, — мягко ответил Квиррелл. — Я встретил его, когда путешествовал по миру. Я был молод, глуп и полон нелепых представлений о добре и зле. Лорд Волан-де-Морт показал мне, как сильно я заблуждался. Добра и зла не существует — есть только сила, есть только власть, и есть те, кто слишком слаб, чтобы стремиться к ней… С тех пор я служу ему верой и правдой, хотя, к сожалению, я не раз подводил его. Ему приходилось быть со мной суровым…
   Квиррелл внезапно поежился.
   — Он не склонен прощать ошибки. Когда мне не удалось украсть камень из «Гринготтса», он был очень мной недоволен. Он наказал меня… Он решил, что должен пристальнееследить за мной и постоянно контролировать меня…
   Голос Квиррелла поплыл по комнате, постепенно затихая. А Гарри вспомнил тот день, когда они с Хагридом были в Косом переулке. Господи, как он мог быть так глуп? Ведь в тот день он видел там Квиррелла в «Дырявом котле». Значит, он 

## 5. Теперь прогоним вопросы через RAG

In [ ]:
# Чтение Excel-файла
df = pd.read_excel(curr_model + '.xlsx')
# Добавление новой колонки для ответов RAG
df['RAG_'+ curr_model] = ''

# Прогон вопросов через RAG и запись ответов в DataFrame
for index, row in df.iterrows():
    question = row['Вопрос']
    rag_answer = make_rag(question)
    df.at[index, 'RAG_'+ curr_model] = rag_answer

# Сохранение DataFrame обратно в Excel
df.to_excel('rag_'+curr_model+'.xlsx', index=False)